# Using ASE simulations with NanoVer

This notebook demonstrates how to set up a molecular simulation with ASE and combine it with NanoVer to run as interactive molecular dynamics (iMD) simulation that can be viewed in the NanoVer iMD-XR client.

## ASE simulation setup

First we set up an ASE simulation (example adapted from [ASE documentation](https://docs.ase-lib.org/examples_generated/03-tutorials/md.html)). NanoVer works with the final `MolecularDynamics` object from ASE, and should be compatible with any combination of [calculators](https://ase.gitlab.io/ase/ase/calculators/calculators.html#module-ase.calculators) and [dynamics](https://ase.gitlab.io/ase/ase/md.html) that ASE supports.

In [1]:
from ase import units

LATTICE_SIZE = 2
TIMESTEP = 1 * units.fs
TEMPERATURE_K = 300
FRICTION = 0.5

In [2]:
from ase.lattice.cubic import FaceCenteredCubic as LatticeFCC
from ase.calculators.emt import EMT
from ase.md.velocitydistribution import thermalize_momenta

# set up initial positions of Cu atoms on Fcc crystal lattice
atoms = LatticeFCC(
    directions=[[1, 0, 0], [0, 1, 0], [0, 0, 1]],
    symbol='Cu',
    size=(LATTICE_SIZE, LATTICE_SIZE, LATTICE_SIZE),
    pbc=True,
)

# describe the interatomic interactions with Effective Medium Theory
atoms.calc = EMT()

# set the initial velocities from Maxwell Boltzmann Distribution
thermalize_momenta(atoms, temperature_K=TEMPERATURE_K)

In [3]:
from ase.md import Langevin

dynamics = Langevin(atoms, timestep=TIMESTEP, temperature_K=TEMPERATURE_K, friction=FRICTION)

C:\Users\ragzo\Documents\REPOS\nanover-server-py-uv\.venv\Lib\site-packages\ase\md\langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


## NanoVer simulation and server setup

Next we wrap the ASE dynamics in a NanoVer simulation (this provides iMD support and compatibility with NanoVer), and start a server providing access to the simulation over the network.

In [4]:
from nanover.ase import ASESimulation

ase_sim = ASESimulation.from_ase_dynamics(dynamics)
ase_sim.frame_interval = 1  # report every simulation step to clients

In [5]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(ase_sim, port=0, name="ase lattice example")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "ase lattice example" (ws://localhost:63012), discoverable on all interfaces on port 54545
Available simulations:
[0]: "Unnamed ASE OpenMM Simulation"
Switched to [0]: "Unnamed ASE OpenMM Simulation"
Switched to [0]: "Unnamed ASE OpenMM Simulation"


## Interacting in virtual reality

Finally, we connect using the [NanoVer iMD-XR client](https://irl2.github.io/nanover-docs/installation.html#installing-the-imd-xr-client) to see and interact with the live ASE dynamics.

<video src="../../notebooks/figures/ase-cu-lattice.webm" controls>